<a href="https://colab.research.google.com/github/Yahir-7/Data_Center_GeoPandas/blob/main/all_data_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Making one CSV for all data gathered
# Heat + drought + County Health Rankings data by county

# Data sources used:
# Extreme Heat: https://web.archive.org/web/20250121194843/https:/www.epa.gov/ejscreen/ejscreen-map-descriptions#clim
# Drought Severity: https://resilience-fema.hub.arcgis.com/maps/FEMA::national-risk-index-annualized-frequency-drought/about
# County Health Rankings: https://www.countyhealthrankings.org/
# Demographic: https://data.census.gov/all
# County boundaries: https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_20m.zip

# Run once if needed
# !pip install geopandas pandas openpyxl pyogrio -q

import pandas as pd
import geopandas as gpd
import glob

# Load county shapefile from Census
counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_20m.zip"
).to_crs(epsg=4326)

# Make FIPS a 5 digit string
counties["FIPS"] = counties["GEOID"].astype(str).str.zfill(5)

# Remove Alaska, Hawaii, and territories
remove_statefps = ["02", "15", "60", "66", "69", "72", "78"]
counties = counties[~counties["STATEFP"].isin(remove_statefps)].copy()

# Start the main dataframe
master = counties[["FIPS", "STATEFP", "NAME", "geometry"]].copy()
master = master.rename(columns={"NAME": "County"})

# Helper function to find files
def find_file(filename):
    matches = glob.glob(f"**/{filename}", recursive=True)
    return matches[0] if len(matches) > 0 else None

# Extreme heat data
heat_path = find_file("EJScreen_HI.shp")

if heat_path is None:
    raise FileNotFoundError("EJScreen_HI.shp was not found.")

heat = gpd.read_file(heat_path).to_crs(epsg=4326)

heat_col = "Average_Da"

heat_points = heat.to_crs(epsg=5070).copy()
heat_points["geometry"] = heat_points.geometry.centroid
heat_points = heat_points.to_crs(epsg=4326)

heat_joined = gpd.sjoin(
    heat_points[[heat_col, "geometry"]],
    master[["FIPS", "geometry"]],
    how="inner",
    predicate="within"
)

heat_county = heat_joined.groupby("FIPS")[heat_col].mean().reset_index()
heat_county = heat_county.rename(columns={heat_col: "extreme_heat"})

master = master.merge(
    heat_county,
    on="FIPS",
    how="left"
)

# Drought data
drought_path = find_file("Census_Tract.shp")

if drought_path is None:
    raise FileNotFoundError("Census_Tract.shp was not found.")

drought = gpd.read_file(drought_path).to_crs(epsg=4326)

drought_col = "DRGT_AFREQ"

drought_points = drought.to_crs(epsg=5070).copy()
drought_points["geometry"] = drought_points.geometry.centroid
drought_points = drought_points.to_crs(epsg=4326)

drought_joined = gpd.sjoin(
    drought_points[[drought_col, "geometry"]],
    master[["FIPS", "geometry"]],
    how="inner",
    predicate="within"
)

drought_county = drought_joined.groupby("FIPS")[drought_col].mean().reset_index()
drought_county = drought_county.rename(columns={drought_col: "drought_severity"})

master = master.merge(
    drought_county,
    on="FIPS",
    how="left"
)

# County Health Rankings data
chr_file = find_file("2025 County Health Rankings Data - v4.xlsx")

if chr_file is not None:
    chr_data = pd.read_excel(
        chr_file,
        sheet_name="Select Measure Data",
        header=1
    )

    chr_data = chr_data[chr_data["County"].notna()].copy()

    chr_data["FIPS"] = (
        chr_data["FIPS"]
        .astype(int)
        .astype(str)
        .str.zfill(5)
    )

    chr_columns = {
        "Average Daily PM2.5": "average_daily_pm25",
        "Presence of Water Violation": "presence_of_water_violation",
        "% Severe Housing Problems": "percent_severe_housing_problems",
        "% Households with Broadband Access": "percent_households_broadband",
        "% Unemployed": "percent_unemployed",
        "Income Ratio": "income_inequality_ratio",
        "Electricity Price": "electricity_price",
        "Gas Price": "gas_price",
        "Nighttime Light Pollution": "nighttime_light_pollution",
        "Existing Noise Exposure": "existing_noise_exposure"
    }

    keep_cols = ["FIPS"]

    for old_col in chr_columns:
        if old_col in chr_data.columns:
            keep_cols.append(old_col)

    chr_clean = chr_data[keep_cols].copy()
    chr_clean = chr_clean.rename(columns=chr_columns)

    master = master.merge(
        chr_clean,
        on="FIPS",
        how="left"
    )

# Demographic data
demo_file = find_file("ACSST1Y2024.S0101-2026-06-18T143622.xlsx")

if demo_file is not None:
    demo_raw = pd.read_excel(
        demo_file,
        sheet_name="Data",
        header=None
    )

    demo_check = str(demo_raw.iloc[0, 1])

    if demo_check != "United States":
        pass

# Save final CSV
final_csv = master.drop(columns="geometry").copy()

output_file = "yahir_geopandas_all-datasets.csv"

final_csv.to_csv(output_file, index=False)